# Garbage Classification — Training
CSE 4830 Group 9

**Run all cells top to bottom.** Total runtime ~35 min on T4 GPU.

**Pre-requisite:** `archive.zip` must be in the root of your Google Drive.

In [ ]:
!git clone https://github.com/LevinMathew1/Garbage-CNN.git /content/Garbage-CNN 2>/dev/null || echo 'Already cloned'
%cd /content/Garbage-CNN
!pip install -q timm
print('Setup complete.')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — go to Runtime > Change runtime type > T4')

In [ ]:
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

zip_src     = Path('/content/drive/MyDrive/archive.zip')
extract_dir = Path('/content/data')

if not zip_src.exists():
    raise FileNotFoundError('archive.zip not found in Google Drive root. Upload it to drive.google.com first.')

if not extract_dir.exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_src) as zf:
        zf.extractall(extract_dir)
    print('Done.')
else:
    print('Dataset already extracted.')

In [ ]:
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def find_data_dir(root):
    for candidate in [root] + sorted([p for p in root.iterdir() if p.is_dir()]):
        subdirs = sorted([p for p in candidate.iterdir() if p.is_dir()])
        if not subdirs:
            continue
        if set(s.name for s in subdirs) >= {'train'}:
            return candidate
        if any(f.suffix.lower() in IMG_EXTS for f in list(subdirs[0].iterdir())[:10]):
            return candidate
    raise RuntimeError(f'Cannot find class directories under {root}')

DATA_DIR = find_data_dir(Path('/content/data'))
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
import subprocess, sys

def run(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('Exit code:', proc.returncode)

## Train CustomCNN (baseline)
Trains from scratch — ~15 min on T4. Target: ≥70% accuracy.

In [ ]:
run([
    sys.executable, 'src/train.py',
    '--model',       'custom_cnn',
    '--data-dir',    str(DATA_DIR),
    '--epochs',      '30',
    '--batch-size',  '64',
    '--lr',          '1e-3',
    '--num-workers', '2',
])

## Train MobileNetV2 (transfer learning)
Backbone frozen for first 5 epochs, then fine-tuned — ~20 min on T4. Target: ≥88% accuracy.

In [ ]:
run([
    sys.executable, 'src/train.py',
    '--model',       'mobilenet_v2',
    '--data-dir',    str(DATA_DIR),
    '--epochs',      '25',
    '--batch-size',  '64',
    '--lr',          '1e-3',
    '--num-workers', '2',
])

## Evaluate both models

In [ ]:
for model in ['custom_cnn', 'mobilenet_v2']:
    print(f'\n{"="*60}\nEvaluating {model}\n{"="*60}')
    run([
        sys.executable, 'src/evaluate.py',
        '--model',       model,
        '--data-dir',    str(DATA_DIR),
        '--num-workers', '2',
    ])

In [ ]:
run([sys.executable, 'scripts/compare_models.py'])

## Results

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for model in ['custom_cnn', 'mobilenet_v2']:
    for fname in [f'{model}_curves.png', f'{model}_confusion_matrix.png']:
        p = Path(f'outputs/figures/{fname}')
        if p.exists():
            print(fname)
            display(Image(str(p)))

In [ ]:
comparison = Path('outputs/metrics/comparison.md')
if comparison.exists():
    print(comparison.read_text())

## Save outputs to Google Drive

In [ ]:
import shutil
from pathlib import Path

dst = Path('/content/drive/MyDrive/garbage-cnn-outputs')
dst.mkdir(parents=True, exist_ok=True)

for folder in ['outputs/figures', 'outputs/metrics', 'outputs/checkpoints']:
    src = Path(folder)
    if src.exists():
        shutil.copytree(src, dst / src.name, dirs_exist_ok=True)
        print(f'Saved {folder}')

print('\nAll outputs saved to Drive: garbage-cnn-outputs/')